검증세트: 테스트 세트를 사용하지 않고 훈련세트를 또 나눈 데이터


---

ex. 전체 데이터 중 20%를 테스트 세트로, 나머지 80%를 훈련세트로 만듦.

훈련세트로 모델 훈련/ 검증세트로 모델 평가

In [1]:
import pandas as pd
wine = pd.read_csv('https://bit.ly/wine_csv_data')

In [2]:
#class열을 타깃으로 나머지 열은 특성으로
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']

In [4]:
#훈련세트와 테스트 세트로 나누기
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state = 42) #train_input의 20%를 val_input으로 만듦

검증세트 만들기

In [6]:
 # train_input과 train_target을 다시 훈련 세트(sub_input, sub_target)와 검증 세트(val_input, val_target)로 20%의 비율로 분할
sub_input, val_input, sub_target, val_target = train_test_split(train_input, train_target, test_size= 0.2, random_state = 42)

In [7]:
print(sub_input.shape, val_input.shape) #훈련세트와 검증세트 크기 확인

(4157, 3) (1040, 3)


In [8]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target) #fit()를 사용해 의사결정 트리 모델 훈련
#sub_input: 훈련에 사용될 특성 데이터
#sub_target: 해당 특성 데이터의 정답
print(dt.score(sub_input, sub_target)) #훈련 모델이 훈련세트에 대해 얼마나 정확한지
print(dt.score(val_input, val_target)) #훈련 모델이 검증세트에 대해 얼마나 정확한지
#훈련세트>검증세트; 과대적합

0.9971133028626413
0.864423076923077


교차검증

In [9]:
from sklearn.model_selection import cross_validate
scores = cross_validate(dt, train_input, train_target) #검증세트를 나누기 전 훈련세트로 교차 검증 진행
print(scores) #scores의 딕셔너리 내용 출력

{'fit_time': array([0.01106501, 0.01256347, 0.01911306, 0.03286171, 0.02601337]), 'score_time': array([0.0033052 , 0.00302434, 0.00937939, 0.00723052, 0.00914264]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [10]:
import numpy as np
print(np.mean(scores['test_score'])) #교차 검증의 최종 점수

0.855300214703487


In [14]:
from sklearn.model_selection import StratifiedKFold #타깃 클래스를 골고루 나누기 위해서
# StratifiedKFold를 사용하여 교차 검증을 수행.
#StratifiedKFold: 각 폴드에 타깃 클래스의 비율이 원본 데이터셋과 동일하게 유지되도록 함
scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
# StratifiedKFold가 적용된 교차 검증 결과의 평균 테스트 점수를 출력
print(np.mean(scores['test_score']))

0.855300214703487


In [16]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42) #n_splits는 몇차 폴드 교차 검증할지 정함.
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score']))

0.8574181117533719


하이퍼파라미터: 모델이 학습할 수 없어 사용자가 지정해야만하는 파라미터


---
클래스나 메서드의 매개변수로 표현



In [17]:
#매개변수 최적값 찾기
from sklearn.model_selection import GridSearchCV
params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

In [18]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1) #탐색 대상 모델과 params 변수 전달하여 그리드 서치 객체

In [19]:
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

In [20]:
dt = gs.best_estimator_
print(dt.score(train_input, train_target))

0.9615162593804117


In [21]:
print(gs.best_params_) #그리드 서치로 찾은 최적의 매개변수

{'min_impurity_decrease': 0.0001}


In [22]:
print(gs.cv_results_['mean_test_score']) #5번의 교차검증으로 얻은 점수

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


In [23]:
print(gs.cv_results_['params'][gs.best_index_]) #최상의 검증을 만든 매개변수 조합

{'min_impurity_decrease': 0.0001}


In [24]:
params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001), #노드를 분할하기 위해 필요한 최소한의 불순도 감소량
          #0.0001부터 0.0009까지 0.0001 간격
          'max_depth': range(5, 20, 1),
          'min_samples_split': range(2, 100, 10) #노드를 분할하기 위해 필요한 최소 샘플 개수
          }

In [25]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1) #그리드 서치 진행
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(5, 20),
                         'min_impurity_decrease': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009]),
                         'min_samples_split': range(2, 100, 10)})

In [26]:
print(gs.best_params_) #최상의 매개변수 조합 확인

{'max_depth': 14, 'min_impurity_decrease': np.float64(0.0004), 'min_samples_split': 12}


In [27]:
print(np.max(gs.cv_results_['mean_test_score'])) #최상의 교차 검증 점수

0.8683865773302731


랜덤서치: 매개변수를 샘플링할 수 있는 확률분포 객체 전달

In [28]:
from scipy.stats import uniform, randint #균등분포에서 샘플링한다.
#randint: 정숫값 #uniform: 실숫값

In [29]:
rgen = randint(0, 10)
rgen.rvs(10) #10개의 숫자 샘플링

array([9, 4, 5, 8, 5, 5, 3, 9, 6, 7])

In [30]:
np.unique(rgen.rvs(1000), return_counts= True)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([ 91, 104, 100, 121, 102,  92,  95, 111,  87,  97]))

In [31]:
ugen = uniform(0, 1)
ugen.rvs(10) #0~1 사이 10개의 실수 추출

array([0.80709534, 0.70626551, 0.88089573, 0.13419462, 0.45075356,
       0.29098293, 0.88765847, 0.79722515, 0.85985519, 0.50254506])

In [33]:
#min_samples_leaf: 리프가되기 위한 최소 샘플의 개수
params = {'min_impurity_decrease': uniform(0.0001, 0.001), #0.0001 과 0.001 사이의 실수값 샘플링
          'max_depth': randint(20, 50), # 20과 50 사이의 정수 샘플링
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25)
          }

In [35]:
from sklearn.model_selection import RandomizedSearchCV
rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params, n_iter = 100, n_jobs = -1, random_state = 42)
rs.fit(train_input, train_target) #params 에서 정의된 매개변수 범위에서 총 100번 샘플링해 교차 검증 수행, 최적의 매개변수 조합

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7ade9e8f0710>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7ade9e8f30b0>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7ade9e8f3830>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7ade9e8f3560>},
                   random_state=42)

In [36]:
print(rs.best_params_) #최적의 매개변수 조합

{'max_depth': 39, 'min_impurity_decrease': np.float64(0.00034102546602601173), 'min_samples_leaf': 7, 'min_samples_split': 13}


In [37]:
print(np.max(rs.cv_results_['mean_test_score'])) #최고의 교차검증 점수

0.8695428296438884


In [38]:
dt = rs.best_estimator_
print(dt.score(test_input, test_target)) #best_estimtor_ 속성에 최종 모델 결정하여 테스트 세트 성능 확인

0.86


**질문사항: StratifiedKFold의 역할 코드에서 교차 검증 시 StratifiedKFold를 사용했는데 일반적인 KFold와 차이가 궁금합니다. '타깃 클래스 비율을 유지한다'는 점이 분류 문제에서 왜 중요한가요?**